In [ ]:
%%sql -r dataframe_1
CREATE DATABASE IF NOT EXISTS WIKIMEDIA_DB;
USE WIKIMEDIA_DB;

CREATE OR REPLACE STORAGE INTEGRATION azure_adls_snowflake_int
  TYPE = EXTERNAL_STAGE
  STORAGE_PROVIDER = 'AZURE'
  ENABLED = TRUE
  AZURE_TENANT_ID = '***************'
  STORAGE_ALLOWED_LOCATIONS = ('azure://flightstacc.blob.core.windows.net/bronze/');

DESC STORAGE INTEGRATION azure_adls_snowflake_int;

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE FILE FORMAT parquet_format
  TYPE = PARQUET;

CREATE OR REPLACE STAGE adls_batch_stage
  STORAGE_INTEGRATION = azure_adls_snowflake_int
  URL = 'azure://flightstacc.blob.core.windows.net/bronze/snowflake_staging_batch/batch_data/'
  FILE_FORMAT = parquet_format;

LIST @adls_batch_stage;

In [ ]:
%%sql -r dataframe_5
CREATE OR REPLACE FILE FORMAT parquet_format
  TYPE = PARQUET;

CREATE OR REPLACE STAGE adls_api_stage
  STORAGE_INTEGRATION = azure_adls_snowflake_int
  URL = 'azure://flightstacc.blob.core.windows.net/bronze/snowflake_staging_api/stream_data/'
  FILE_FORMAT = parquet_format;

LIST @adls_api_stage;

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE TABLE wikiMEDIA_batch_data AS 
SELECT 
    $1:project::STRING AS project,
    $1:access:: STRING AS access,
    $1:year::INT as year,
    $1:month::INT as month,
    $1:day::STRING() as day,
    $1:article::STRING As article,
    $1:views::INT AS views,
    $1:rank::INT as rank 
FROM @adls_batch_stage
(pattern=> '.*\.parquet');

SELECT * FROM wikimedia_batch_data LIMIT 10;

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE TASK WIKIMEDIA_DB.PUBLIC.wikimedia_stream_task
  WAREHOUSE = COMPUTE_WH
  SCHEDULE = '720 MINUTES'
AS
  COPY INTO WIKIMEDIA_DB.PUBLIC.wikimedia_stream_data
  FROM (
    SELECT 
      $1:sequence_number::INT,
      $1:enqueued_time::STRING,
      $1:id::INT,
      $1:timestamp::STRING,
      $1:title::STRING,
      $1:bot::BOOLEAN,
      $1:type::STRING,
      $1:user::STRING,
      $1:wiki::STRING
    FROM @WIKIMEDIA_DB.PUBLIC.adls_api_stage
  )
  FILE_FORMAT = (FORMAT_NAME = 'parquet_format');

ALTER TASK WIKIMEDIA_DB.PUBLIC.wikimedia_stream_task RESUME;
